# Proyecto Big Data - Steam Reviews (Modin + Ray)


In [1]:
# ==========================================================
# CONFIGURACION DE RECOLECCION DE TIEMPOS
# ==========================================================
# Arquitectura B: 1 Master + 4 Workers (n2-standard-4)

tiempos_resultados = {}
arquitectura = "4_workers"


In [2]:
!pip install "modin[ray]==0.26.1" "ray[default]==2.55.1" gcsfs fsspec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 94.6 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 118.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 86.2 MB/s  0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.5
    Uninstalling pydantic-2.13.5:
      Successfully uninstalled pydantic-2.13.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [opencensus]1 [ray]n]ic]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-profiling 3.0.0 requires tangled-up-in-unicode==0.1.0, but you have tangled-up-in-unicode 0.2.0 which is incompatible.


In [3]:
!gsutil cp gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv /tmp/steam_reviews_500k.csv

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv...
/ [1 files][667.7 MiB/667.7 MiB]                                                
Operation completed over 1 objects/667.7 MiB.                                    


In [4]:
# Prender Ray
import ray

# Por si quedo una instancia de Ray corriendo de una ejecucion anterior
# (ej. reejecutaste esta celda sin reiniciar el kernel)
ray.shutdown()

# NOTA IMPORTANTE (arquitectura distribuida):
# Sin 'address', ray.init() crea un cluster de Ray LOCAL usando solo
# los recursos del nodo master. NO se conecta automaticamente a los
# workers de Dataproc. Es decir: tal como esta, Modin no aprovecha
# los nodos worker de la Arquitectura A/B, todo corre en el master.
# Si necesitas que Modin use realmente el cluster (2 o 4 workers),
# hay que levantar un Ray cluster real sobre Dataproc y conectarse
# con ray.init(address="auto") o ray.init(address="ray://<head-ip>:10001").
# Para el alcance de este proyecto, documentar esta limitacion es valido.

# Corre '!free -h' en una celda antes para ver la RAM libre real del
# nodo y ajustar estos topes con margen.
ray.init(
    ignore_reinit_error=True,
    object_store_memory=2 * 1024**3,  # tope del object store
    _memory=6 * 1024**3                # tope de memoria para tareas Ray
)
#Apagar ray
#import ray

#ray.shutdown()


2026-09-21 15:43:37,978	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/opt/conda/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/ray/_private/worker.py:2066: FutureWarning: Pydantic v1 is deprecated and will no longer be supported in Ray 2.56. Please upgrade to Pydantic v2 by running `pip install pydantic>=2`. See https://github.com/ray-project/ray/issues/58876 for more details.
  warnings.warn(


Python version:,3.11.14
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


In [5]:
import modin.pandas as mpd
dfm = mpd.read_csv("/tmp/steam_reviews_500k.csv")
print(dfm.shape)

(500000, 24)


## Consulta 1 - Exploración y validación del dataset

Objetivo: conocer dimensiones, estructura, tipos de datos y valores nulos del dataset.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [6]:
# ==========================================================
# CONSULTA 1 - MODIN
# ==========================================================

import time

inicio = time.time()

print("===== MODIN =====")


print("\nDimensiones del dataset")

print("Filas:", dfm.shape[0])

print("Columnas:", dfm.shape[1])


print("\nEstructura de datos")

print(dfm.dtypes)


print("\nValores nulos")

print(
    dfm.isnull()
       .sum()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_1"] = fin - inicio


===== MODIN =====

Dimensiones del dataset
Filas: 500000
Columnas: 24

Estructura de datos
recommendationid                    int64
appid                               int64
game                               object
author_steamid                      int64
author_num_games_owned              int64
author_num_reviews                  int64
author_playtime_forever             int64
author_playtime_last_two_weeks      int64
author_playtime_at_review           int64
author_last_played                  int64
language                           object
review                             object
timestamp_created                   int64
timestamp_updated                   int64
voted_up                            int64
votes_up                            int64
votes_funny                         int64
weighted_vote_score               float64
comment_count                       int64
steam_purchase                      int64
received_for_free                   int64
written_during_early_access

## Consulta 2 - Eliminación de duplicados

Objetivo: eliminar registros repetidos considerando author_steamid, appid y review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [7]:
# ==========================================================
# CONSULTA 2 - ELIMINACIÓN DE DUPLICADOS CON MODIN
# ==========================================================

import time

inicio = time.time()

print("===== MODIN =====")


registros_iniciales = len(dfm)


print("Registros iniciales:",
      registros_iniciales)


# Eliminación de duplicados

dfm_clean = dfm.drop_duplicates(
    subset=[
        "author_steamid",
        "appid",
        "review"
    ]
)


registros_finales = len(dfm_clean)


print("Registros después de eliminar duplicados:",
      registros_finales)


print("Duplicados eliminados:",
      registros_iniciales - registros_finales)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_2"] = fin - inicio


===== MODIN =====
Registros iniciales: 500000
Registros después de eliminar duplicados: 315809
Duplicados eliminados: 184191

Tiempo de ejecución: 8.99345588684082 segundos


## Consulta 3 - Tratamiento de valores nulos

Objetivo: identificar valores faltantes y limpiar registros sin información en review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [8]:
# ==========================================================
# CONSULTA 3 - TRATAMIENTO DE VALORES NULOS CON MODIN
# ==========================================================

import time

inicio = time.time()

print("===== MODIN =====")


registros_iniciales = len(dfm_clean)


print("Registros iniciales:",
      registros_iniciales)


print("\nValores nulos antes:")

print(
    dfm_clean.isnull()
            .sum()
)


# Eliminación de registros sin review

dfm_null_clean = dfm_clean.dropna(
    subset=["review"]
)


registros_finales = len(dfm_null_clean)


print("\nRegistros después de limpiar:",
      registros_finales)


print("Registros eliminados:",
      registros_iniciales - registros_finales)


print("\nValores nulos después:")

print(
    dfm_null_clean.isnull()
                  .sum()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_3"] = fin - inicio


===== MODIN =====
Registros iniciales: 315809

Valores nulos antes:
recommendationid                       0
appid                                  0
game                                  22
author_steamid                         0
author_num_games_owned                 0
author_num_reviews                     0
author_playtime_forever                0
author_playtime_last_two_weeks         0
author_playtime_at_review              0
author_last_played                     0
language                               0
review                                 0
timestamp_created                      0
timestamp_updated                      0
voted_up                               0
votes_up                               0
votes_funny                            0
weighted_vote_score                    0
comment_count                          0
steam_purchase                         0
received_for_free                      0
written_during_early_access            0
hidden_in_steam_china         

## Consulta 4 - Transformación de variables

Objetivo: crear review_length como cantidad de caracteres de cada reseña.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [9]:
# ==========================================================
# CONSULTA 4 - TRANSFORMACIÓN DE VARIABLES CON MODIN
# ==========================================================

import time

inicio = time.time()

print("===== MODIN =====")


dfm_transform = dfm_null_clean.copy()


# Crear variable review_length

dfm_transform["review_length"] = (
    dfm_transform["review"]
    .str.len()
)


print("Registros procesados:",
      len(dfm_transform)
)


print("\nEjemplo de transformación:")

print(
    dfm_transform[
        [
            "review",
            "review_length"
        ]
    ].head()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_4"] = fin - inicio


===== MODIN =====
Registros procesados: 315809

Ejemplo de transformación:
                                              review  review_length
0  Don't get me wrong, I love Horizon games, I've...           2184
1                                        add sex pls             11
2  Everyone is hating on this game and I understa...           2908
3  Larguei o jogo no momento que vi a loja que ve...            155
4  50 lirayken bile millet almıyordu kaç yıllık o...             83

Tiempo de ejecución: 0.5908999443054199 segundos


## Consulta 5 - Filtrado de reseñas recomendadas

Objetivo: seleccionar registros donde voted_up sea igual a 1.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [10]:
# ==========================================================
# CONSULTA 5 - FILTRADO DE RESEÑAS RECOMENDADAS CON MODIN
# ==========================================================

import time

inicio = time.time()

print("===== MODIN =====")


registros_iniciales = len(dfm_transform)


print("Registros iniciales:",
      registros_iniciales)



# Filtrar reseñas recomendadas

dfm_positive = dfm_transform[
    dfm_transform["voted_up"] == 1
]


registros_finales = len(dfm_positive)


print("Registros recomendados:",
      registros_finales)


print("Porcentaje de recomendaciones:",
      (registros_finales / registros_iniciales) * 100,
      "%")


print("\nEjemplo de datos:")

print(
    dfm_positive[
        [
            "game",
            "review",
            "voted_up"
        ]
    ].head()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_5"] = fin - inicio


===== MODIN =====
Registros iniciales: 315809
Registros recomendados: 266640
Porcentaje de recomendaciones: 84.43077936347603 %

Ejemplo de datos:
                          game  \
1                   Obama Maze   
2             Railway Empire 2   
4                     RimWorld   
5            Sword and Fairy 3   
6  Majikoi! Love Me Seriously!   

                                              review  voted_up  
1                                        add sex pls         1  
2  Everyone is hating on this game and I understa...         1  
4  50 lirayken bile millet almıyordu kaç yıllık o...         1  
5  就当为以前的盗版补票吧。不过18年前的游戏直接原版扔上来这样好吗？不好吧。不重置一下吗？没有...         1  
6  Antes de extenderme y si nadie quiere leer, pu...         1  

Tiempo de ejecución: 5.303575277328491 segundos


## Consulta 6 - Cantidad de reseñas por videojuego

Objetivo: agrupar por game y calcular la cantidad total de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [11]:
# ==========================================================
# CONSULTA 6 - CANTIDAD DE RESEÑAS POR VIDEOJUEGO - MODIN
# ==========================================================

import time

inicio = time.time()

print("===== MODIN =====")


dfm_game_reviews = (
    dfm_transform
    .groupby("game")
    .size()
    .reset_index(
        name="total_reviews"
    )
    .sort_values(
        "total_reviews",
        ascending=False
    )
)


print("Cantidad de videojuegos procesados:",
      len(dfm_game_reviews))


print("\nTop videojuegos por cantidad de reseñas:")

print(
    dfm_game_reviews.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_6"] = fin - inicio


===== MODIN =====
Cantidad de videojuegos procesados: 22991

Top videojuegos por cantidad de reseñas:
                                 game  total_reviews
3718                 Counter-Strike 2           1923
13300             PUBG: BATTLEGROUNDS           1513
17029                  Stardew Valley           1413
18107                        Terraria           1264
19222        The Witcher 3: Wild Hunt           1228
7820               Grand Theft Auto V           1228
19523  Tom Clancy's Rainbow Six Siege           1224
18499                      The Forest           1168
20826                Wallpaper Engine           1093
15284                            Rust           1034

Tiempo de ejecución: 1.3636703491210938 segundos


## Consulta 7 - Porcentaje de recomendación por videojuego

Objetivo: calcular porcentaje de reseñas positivas por juego usando voted_up.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [12]:
# ==========================================================
# CONSULTA 7 - PORCENTAJE DE RECOMENDACIÓN POR VIDEOJUEGO
# MODIN
# ==========================================================

import time


inicio = time.time()

print("===== MODIN =====")


dfm_recommendation = (
    dfm_transform
    .groupby("game")
    .agg(
        total_reviews=("voted_up", "count"),
        positive_reviews=("voted_up", "sum")
    )
    .reset_index()
)


dfm_recommendation["recommendation_percentage"] = (
    dfm_recommendation["positive_reviews"] /
    dfm_recommendation["total_reviews"]
    * 100
)


dfm_recommendation = (
    dfm_recommendation
    .sort_values(
        "recommendation_percentage",
        ascending=False
    )
)


print("Videojuegos procesados:",
      len(dfm_recommendation))


print("\nTop juegos recomendados:")

print(
    dfm_recommendation.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_7"] = fin - inicio


===== MODIN =====
Videojuegos procesados: 22991

Top juegos recomendados:
                                                    game  total_reviews  \
22990                                   🧠 OUT OF THE BOX              1   
0       Devil May Cry 5 Vergil's Rebirth Sound Selection              5   
1                   Fishing Planet: Congo Discovery Pack              1   
2                Hogwarts Legacy: Dark Arts Garrison Hat              2   
3                                               Marfusha              9   
5                            神明的一天世界-God's One Day World              2   
7                                               !Anyway!              1   
9                                             #CuteSnake              1   
10                                          #CuteSnake 2              1   
11                                       #KILLALLZOMBIES              1   

       positive_reviews  recommendation_percentage  
22990                 1                      10

## Consulta 8 - Promedio de horas jugadas por videojuego

Objetivo: calcular promedio de author_playtime_forever por juego.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [13]:
# ==========================================================
# CONSULTA 8 - PROMEDIO DE HORAS JUGADAS POR VIDEOJUEGO
# MODIN
# ==========================================================

import time


inicio = time.time()

print("===== MODIN =====")


dfm_playtime = (
    dfm_transform
    .groupby("game")
    ["author_playtime_forever"]
    .mean()
    .reset_index()
)


dfm_playtime = (
    dfm_playtime
    .rename(
        columns={
            "author_playtime_forever":
            "avg_playtime_minutes"
        }
    )
    .sort_values(
        "avg_playtime_minutes",
        ascending=False
    )
)


print("Videojuegos procesados:",
      len(dfm_playtime))


print("\nVideojuegos con mayor promedio de juego:")

print(
    dfm_playtime.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_8"] = fin - inicio


===== MODIN =====
Videojuegos procesados: 22991

Videojuegos con mayor promedio de juego:
                                     game  avg_playtime_minutes
12936       Oops!!! I Slept With Your Mom             2181697.0
10339                 Legions of Ashworld             1982834.0
1155                   Aquarium Simulator             1687347.0
20815                            WalkinVR             1366674.0
8851                        Houdini Indie             1264521.5
12811  Oh, you touch my balls ( ͡° ͜ʖ ͡°)             1247696.0
18896     The Putinland: Divide & Conquer             1114523.0
10931                        MachineCraft              987286.0
16590                Solitaire Forever II              945538.0
3919          Crusaders of the Lost Idols              903523.0

Tiempo de ejecución: 1.104006052017212 segundos


## Consulta 9 - Longitud promedio de reseñas por videojuego

Objetivo: calcular promedio de review_length agrupado por game.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [14]:
# ==========================================================
# CONSULTA 9 - LONGITUD PROMEDIO DE RESEÑAS POR VIDEOJUEGO
# MODIN
# ==========================================================

import time


inicio = time.time()

print("===== MODIN =====")


dfm_review_length = (
    dfm_transform
    .groupby("game")
    ["review_length"]
    .mean()
    .reset_index()
)


dfm_review_length = (
    dfm_review_length
    .rename(
        columns={
            "review_length":
            "avg_review_length"
        }
    )
    .sort_values(
        "avg_review_length",
        ascending=False
    )
)


print("Videojuegos procesados:",
      len(dfm_review_length))


print("\nVideojuegos con reseñas más extensas:")

print(
    dfm_review_length.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_9"] = fin - inicio


===== MODIN =====
Videojuegos procesados: 22991

Videojuegos con reseñas más extensas:
                         game  avg_review_length
21108          Wayward Strand             8000.0
9256        Indie Game Battle             8000.0
18636        The Knight Witch             8000.0
11139          Mary Skelter 2             8000.0
14238  Qbeh-1: The Atlas Cube             8000.0
4633             Deadly Flare             7999.0
1574             Azusa Online             7999.0
890            Alterium Shift             7998.0
18313           The Companion             7996.0
5628                 Dynopunk             7996.0

Tiempo de ejecución: 0.683347225189209 segundos


## Consulta 10 - Ranking de videojuegos

Objetivo: ordenar videojuegos considerando porcentaje de recomendación y cantidad de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [15]:
# ==========================================================
# CONSULTA 10 - RANKING DE VIDEOJUEGOS
# MODIN
# ==========================================================

import time


inicio = time.time()

print("===== MODIN =====")


dfm_ranking = (
    dfm_transform
    .groupby("game")
    .agg(
        total_reviews=("voted_up", "count"),
        positive_reviews=("voted_up", "sum")
    )
    .reset_index()
)


dfm_ranking["recommendation_percentage"] = (
    dfm_ranking["positive_reviews"] /
    dfm_ranking["total_reviews"]
    * 100
)


dfm_ranking = (
    dfm_ranking[
        dfm_ranking["total_reviews"] >= 100
    ]
    .sort_values(
        [
            "recommendation_percentage",
            "total_reviews"
        ],
        ascending=False
    )
)


print("Videojuegos rankeados:",
      len(dfm_ranking))


print("\nTop videojuegos:")

print(
    dfm_ranking.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Modin_Consulta_10"] = fin - inicio


===== MODIN =====
Videojuegos rankeados: 609

Top videojuegos:
                                       game  total_reviews  positive_reviews  \
10293                         Left 4 Dead 2            821               821   
17349                            Subnautica            807               807   
11529                                Mirror            604               604   
13791  Plants vs. Zombies: Game of the Year            532               532   
11855                Mount & Blade: Warband            428               428   
8137                                  Hades            390               390   
3717                         Counter-Strike            386               386   
15721                          Satisfactory            384               384   
4231                                   DOOM            382               382   
13384                        Papers, Please            356               356   

       recommendation_percentage  
10293                

In [16]:
# ==========================================================
# EXPORTACION AUTOMATIZADA DE RESULTADOS A GOOGLE CLOUD STORAGE
# ==========================================================

!pip install -q gcsfs fsspec

import pandas as pd

framework = "modin"

df_tiempos = pd.DataFrame(
    list(tiempos_resultados.items()),
    columns=["Consulta", "Tiempo_segundos"]
)

print(df_tiempos)

ruta_salida = f"gs://bigdata-2026-02/proyecto01/tiempos_{framework}_{arquitectura}.csv"

df_tiempos.to_csv(ruta_salida, index=False)

print(f"\nResultados exportados a: {ruta_salida}")


            Consulta  Tiempo_segundos
0   Modin_Consulta_1         0.641603
1   Modin_Consulta_2         8.993456
2   Modin_Consulta_3         2.088817
3   Modin_Consulta_4         0.590900
4   Modin_Consulta_5         5.303575
5   Modin_Consulta_6         1.363670
6   Modin_Consulta_7         1.049051
7   Modin_Consulta_8         1.104006
8   Modin_Consulta_9         0.683347
9  Modin_Consulta_10         1.080676

Resultados exportados a: gs://bigdata-2026-02/proyecto01/tiempos_modin_4_workers.csv
